In [8]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import algoritmos_clasificacion as t
import importlib
importlib.reload(t)

<module 'algoritmos_clasificacion' from '/home/hezek/FING/MachineLearning/MachineLearningLab/LAB1/Script/algoritmos_clasificacion.py'>

In [9]:

# 1. Cargar dataset
S = t.cargarDataset("data_dia")

# 2. Atributos y variable objetivo
X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto","amplitud_carrasco","amplitud_salto",
"dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
"dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom","vis_carrasco_prom","vis_salto_prom"]]

y = S["dia_lluvioso"]

# 3. Train/Test split (80% - 20%), estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=20,
    stratify=y
)

# 4. Configurar validación cruzada
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)

# 5. Comparar criterios
criterios = ["gain", "gini", "gain_ratio"]
resultados = {}

for criterio in criterios:
    print(f"\n=== Evaluando criterio: {criterio.upper()} ===")
    
    # Crear modelo
    model = t.ID3Classifier(min_info_gain=0.003, criterio=criterio)
    
    # Validación cruzada en train
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf) #default scoring accuracy
    resultados[criterio] = cv_scores
    
    print("Scores por fold:", [round(s,4) for s in cv_scores])
    print(f"Accuracy promedio (cada fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Entrenar en todo el train y evaluar en test
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Accuracy
    acc_test = accuracy_score(y_test, y_pred)
    
    # F1-score
    f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si') #pos_label='si' para la clase positiva
    
    # Recall
    recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
    
    print(f"Accuracy en test: {acc_test:.4f}")
    print(f"F1-score en test: {f1_test:.4f}")
    print(f"Recall en test: {recall_test:.4f}")
    print("Matriz de confusión:")
    print(cm)
    
# 6. Resumen final
print("\nRESUMEN DE COMPARACIÓN (Accuracy promedio cross-val ± desviación)")
for criterio, scores in resultados.items():
    print(f"{criterio.upper():<12} : {scores.mean():.4f} ± {scores.std():.4f}")

# 7. Identificar mejor criterio
mejor_criterio = max(resultados.keys(), key=lambda k: resultados[k].mean())
print(f"\nMejor criterio: {mejor_criterio.upper()}") #seguyn accuracy promedio (SOLO)

#matriz
      #pred si  #pred no
#   si [TP   |   FN]
#   no [FP   |   TN]


=== Evaluando criterio: GAIN ===
Scores por fold: [np.float64(0.7038), np.float64(0.6967), np.float64(0.6853), np.float64(0.7016), np.float64(0.7318)]
Accuracy promedio (cada fold): 0.7038 ± 0.0153
Accuracy en test: 0.7000
F1-score en test: 0.2528
Recall en test: 0.2291
Matriz de confusión:
[[ 206  693]
 [ 525 2636]]

=== Evaluando criterio: GINI ===
Scores por fold: [np.float64(0.7457), np.float64(0.7312), np.float64(0.7377), np.float64(0.7641), np.float64(0.7502)]
Accuracy promedio (cada fold): 0.7458 ± 0.0112
Accuracy en test: 0.7431
F1-score en test: 0.1983
Recall en test: 0.1435
Matriz de confusión:
[[ 129  770]
 [ 273 2888]]

=== Evaluando criterio: GAIN_RATIO ===
Scores por fold: [np.float64(0.7414), np.float64(0.7087), np.float64(0.7383), np.float64(0.7354), np.float64(0.7277)]
Accuracy promedio (cada fold): 0.7303 ± 0.0117
Accuracy en test: 0.7298
F1-score en test: 0.2181
Recall en test: 0.1702
Matriz de confusión:
[[ 153  746]
 [ 351 2810]]

RESUMEN DE COMPARACIÓN (Accuracy 

In [10]:
# Lista de valores a probar para min_info_gain
min_info_values = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006]

criterio = "gini"
resultados_gini = {}

for min_gain in min_info_values:
    print(f"\n=== Evaluando GINI con min_info_gain={min_gain} ===")
    
    # Crear modelo con el valor actual de min_info_gain
    model = t.ID3Classifier(min_info_gain=min_gain, criterio=criterio)
    
    # Validación cruzada en train
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
    resultados_gini[min_gain] = cv_scores
    
    print("Scores por fold:", [round(s,4) for s in cv_scores])
    print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Entrenar en todo el train y evaluar en test
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Accuracy
    acc_test = accuracy_score(y_test, y_pred)
    # F1-score
    f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si')
    # Recall
    recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
    
    print(f"Accuracy en test: {acc_test:.4f}")
    print(f"F1-score en test: {f1_test:.4f}")
    print(f"Recall en test: {recall_test:.4f}")
    print("Matriz de confusión:")
    print(cm)

# Resumen final de GINI para los distintos min_info_gain
print("\nRESUMEN GINI (Accuracy promedio CV ± desviación)")
for min_gain, scores in resultados_gini.items():
    print(f"min_info_gain={min_gain:<6} : {scores.mean():.4f} ± {scores.std():.4f}")



=== Evaluando GINI con min_info_gain=0.001 ===
Scores por fold: [np.float64(0.7004), np.float64(0.6875), np.float64(0.6903), np.float64(0.6902), np.float64(0.6911)]
Accuracy promedio CV: 0.6919 ± 0.0044
Accuracy en test: 0.6938
F1-score en test: 0.2579
Recall en test: 0.2403
Matriz de confusión:
[[ 216  683]
 [ 560 2601]]

=== Evaluando GINI con min_info_gain=0.002 ===
Scores por fold: [np.float64(0.7457), np.float64(0.689), np.float64(0.7377), np.float64(0.7139), np.float64(0.7287)]
Accuracy promedio CV: 0.7230 ± 0.0200
Accuracy en test: 0.7429
F1-score en test: 0.1982
Recall en test: 0.1435
Matriz de confusión:
[[ 129  770]
 [ 274 2887]]

=== Evaluando GINI con min_info_gain=0.003 ===
Scores por fold: [np.float64(0.7457), np.float64(0.7312), np.float64(0.7377), np.float64(0.7641), np.float64(0.7502)]
Accuracy promedio CV: 0.7458 ± 0.0112
Accuracy en test: 0.7431
F1-score en test: 0.1983
Recall en test: 0.1435
Matriz de confusión:
[[ 129  770]
 [ 273 2888]]

=== Evaluando GINI con mi

In [11]:

# 1. Cargar dataset
S = t.cargarDataset("data_dia_rf")

# 2. Atributos y variable objetivo
X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto","amplitud_carrasco","amplitud_salto",
"dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
"dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom","vis_carrasco_prom","vis_salto_prom"]]

y = S["dia_lluvioso"]



# 3. Train/Test split (80% - 20%), estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=20,
    stratify=y
)

# 4. Configurar validación cruzada
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)

# Lista de valores a probar (ejemplo: número de árboles)
n_estimators_values = [10, 50, 100, 200]
resultados_rf = {}

for n_est in n_estimators_values:
    print(f"\n=== Evaluando Random Forest con n_estimators={n_est} ===")
    
    # Crear modelo con hiperparámetro actual
    rf_model = RandomForestClassifier(
        n_estimators=n_est,
        random_state=20
    )
    
    # Validación cruzada en train
    cv_scores = cross_val_score(rf_model, X_train, y_train, cv=kf)
    resultados_rf[n_est] = cv_scores
    
    print("Scores por fold:", [round(s,4) for s in cv_scores])
    print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Entrenar en todo el train y evaluar en test
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    
    # Accuracy
    acc_test = accuracy_score(y_test, y_pred)
    # F1-score
    f1_test = f1_score(y_test, y_pred, average='binary')
    # Recall
    recall_test = recall_score(y_test, y_pred, average='binary')
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"Accuracy en test: {acc_test:.4f}")
    print(f"F1-score en test: {f1_test:.4f}")
    print(f"Recall en test: {recall_test:.4f}")
    print("Matriz de confusión:")
    print(cm)

# Resumen final de Random Forest
print("\nRESUMEN RANDOM FOREST (Accuracy promedio CV ± desviación)")
for n_est, scores in resultados_rf.items():
    print(f"n_estimators={n_est:<6} : {scores.mean():.4f} ± {scores.std():.4f}")



=== Evaluando Random Forest con n_estimators=10 ===
Scores por fold: [np.float64(0.7679), np.float64(0.7632), np.float64(0.7703), np.float64(0.7644), np.float64(0.7672)]
Accuracy promedio CV: 0.7666 ± 0.0025
Accuracy en test: 0.7606
F1-score en test: 0.1562
Recall en test: 0.1001
Matriz de confusión:
[[2998  163]
 [ 809   90]]

=== Evaluando Random Forest con n_estimators=50 ===
Scores por fold: [np.float64(0.7783), np.float64(0.7685), np.float64(0.7808), np.float64(0.7758), np.float64(0.7749)]
Accuracy promedio CV: 0.7756 ± 0.0041
Accuracy en test: 0.7722
F1-score en test: 0.1379
Recall en test: 0.0823
Matriz de confusión:
[[3061  100]
 [ 825   74]]

=== Evaluando Random Forest con n_estimators=100 ===
Scores por fold: [np.float64(0.7799), np.float64(0.7688), np.float64(0.7771), np.float64(0.777), np.float64(0.7801)]
Accuracy promedio CV: 0.7766 ± 0.0041
Accuracy en test: 0.7746
F1-score en test: 0.1392
Recall en test: 0.0823
Matriz de confusión:
[[3071   90]
 [ 825   74]]

=== Evalu

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import algoritmos_clasificacion as t
from sklearn.model_selection import KFold

def crear_lags_concat(X, y, nd):
    """
    Genera variables lag (días previos) hasta 'nd' para cada columna de X.
    Devuelve X_lag e y_lag con las filas sincronizadas y sin NaN.
    """
    lags = {}
    for col in X.columns:
        for d in range(1, nd+1):
            lags[f"{col}_lag{d}"] = X[col].shift(d)

    X_lag = pd.concat([X, pd.DataFrame(lags)], axis=1)

    # Eliminar filas con NaN generadas por shift
    mask = ~X_lag.isna().any(axis=1)
    return X_lag[mask], y[mask]


# Hiperparámetros a probar con Bayes
m_values = [0.5, 1, 2, 5]
nd_values = [3] # [1, 2, 3]

resultados_bayes = {}

S = t.cargarDataset("data_dia")

X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
       "amplitud_carrasco","amplitud_salto",
       "dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
       "dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom",
       "vis_carrasco_prom","vis_salto_prom"]]

y = S["dia_lluvioso"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

for m in m_values:
    for nd in nd_values:
        print(f"\n=== Evaluando Naive Bayes con m={m}, nd={nd} ===")
        
        X_train_lag, y_train_lag = crear_lags_concat(X_train, y_train, nd)
        X_test_lag, y_test_lag = crear_lags_concat(X_test, y_test, nd)

        # Crear modelo
        model = t.NaiveBayesClassifier(m=m, nd=nd)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train_lag, y_train_lag, cv=kf)
        resultados_bayes[(m, nd)] = cv_scores
        
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train_lag, y_train_lag)
        y_pred = model.predict(X_test_lag)

        # Accuracy
        acc_test = accuracy_score(y_test, y_pred)
        # F1-score
        f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si')
        # Recall
        recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
        
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)
# Resumen final de Naive Bayes para los distintos m y nd
print("\nRESUMEN NAIVE BAYES (Accuracy promedio CV ± desviación)")
for (m, nd), scores in resultados_bayes.items():
    print(f"m={m}, nd={nd} : {scores.mean():.4f} ± {scores.std():.4f}")


=== Evaluando Naive Bayes con m=0.5, nd=3 ===


NameError: name 'pd' is not defined